In [1]:
import os
from dotenv import load_dotenv

from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from openai import AsyncOpenAI, OpenAI

from openagv.modules.audio import DeepgramAnalyzer

# Load environment variables from .env
load_dotenv()

async_client = AsyncOpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)

openai_client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)

chat_completion_service = OpenAIChatCompletion(
    service_id="my-service-id",
    ai_model_id="openai/gpt-4o-mini",
    async_client=async_client
)



In [2]:
from openagv import AssetBin, SKLoopExecutor, UserInstruction, OTIOTimeline
from openagv.modules.vision import ORVisionAnalyzer

# Mock client for the Vision Analyzer (OpenRouter/Other)
class MockORClient:
    pass

orclient = MockORClient()

# Define the instruction
instruct = UserInstruction("Analyze all the assets we have and print it to debug. Including videos, transcribe them")
# Setup AssetBin and add assets
ab = AssetBin()
ab.add("examples/assets/notaflower.png")
ab.add("examples/assets/softram.webm")

# Initialize modules
vision_analyzer = ORVisionAnalyzer(client=openai_client, model='mistralai/ministral-8b-2512')
transcriber = DeepgramAnalyzer(api_key=os.getenv("DEEPGRAM_API_KEY"))

# Setup Executor
ex = SKLoopExecutor(ab, instruct, chat_completion=chat_completion_service, uses=[vision_analyzer, transcriber], debug=True)

# Execute
await ex.start()


[INFO] Starting real execution with instruction: 'Analyze all the assets we have and print it to debug. Including videos, transcribe them'
[DEBUG] System Prompt: You are a helpful AI assistant capable of analyzing and manipulating media assets.
[DEBUG] Loaded Plugins: ['AssetBin', 'ORVisionAnalyzer', 'DeepgramAnalyzer']
[INFO] Advancing to step: AssetBin.list_assets
[DEBUG] Invoking AssetBin.list_assets with args: {}
[DEBUG] Result from AssetBin.list_assets: ID: 318ab6f580e22b8733ac4176c3e86111afe162c7d38ba92e5b1320c603e816b7 | File: examples/assets/notaflower.png (IMAGE)
ID: 08dbea1db0b18cdf17a280f15249649419aa10854ee340fbc8e3fdea785723c9 | File: examples/assets/softram.webm (VIDEO)
[INFO] Advancing to step: ORVisionAnalyzer.analyze_asset
[DEBUG] Invoking ORVisionAnalyzer.analyze_asset with args: {asset_id='318ab6f580e22b8733ac4176c3e86111afe162c7d38ba92e5b1320c603e816b7'}
[DEBUG] ORVisionAnalyzer analyzing examples/assets/notaflower.png with mistralai/ministral-8b-2512...
[DEBUG] Res

In [3]:
ex.asset_bin.json_dump('dump.json')